# Mancino Market: Predicting Daily Unit Sales
## Complete neural network exercise | Business Analytics and Data Visualization
Prof. Jose Mendoza · Fall 2026

How many units will a store sell in a product category on a future day? Use neural network regression to predict daily unit sales, compare network architectures, and evaluate predictions against a mean-sales baseline. The outcome is a numeric quantity, not a class label.

Mancino Market is a synthetic grocery retailer with five fictional Manhattan stores. One row represents one date, store, and product category. Predictions can inform replenishment planning, but observed sales do not measure unmet demand during stockouts.

### How to use this notebook
1. Upload this `.ipynb` file at [Google Colab](https://colab.research.google.com/) using **File > Upload notebook**. A standard CPU runtime is sufficient; TensorFlow and a GPU are not required.
2. Run sections 1–5 to prepare the data and baseline.
3. Run section 6 to fit networks with hidden layers `(16,)`, `(32,)`, and `(32, 16)`.
4. Compare validation results in section 7, then inspect training curves in section 8.
5. When selection is complete, change `RUN_FINAL_TEST` to `True` in section 9 and run the final test once.

All code is complete. No separate CSV upload is needed when the GitHub connection is available. The data, predictors, and chronological splits match the companion exercises.


## 1. Import the tools
Colab includes these libraries. The notebook prints their versions so results can be traced to the environment used.


In [ ]:
import io
import time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from IPython.display import display
from sklearn.base import clone
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.dummy import DummyRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

SEED = 42
np.random.seed(SEED)
plt.rcParams.update({'figure.figsize': (9, 4), 'axes.spines.top': False,
                     'axes.spines.right': False})
print('pandas:', pd.__version__, '| scikit-learn:', sklearn.__version__)


## 2. Load and inspect the data
Source: [mancino_daily_sales.csv](https://github.com/jrmst102/mancino/blob/main/data/v1_2025-11-01/mancino_daily_sales.csv).
The verified file contains 22,875 rows, covering January 1–November 1, 2025. The code reports the current file dimensions in case the repository changes.

| Field | Meaning | Use in this notebook |
|---|---|---|
| `date` | Selling date | Split data and derive calendar features |
| `store_id` | Store identifier | Categorical predictor |
| `category` | Product category | Categorical predictor |
| `day_of_week` | Weekday | Recomputed from the date |
| `avg_price` | Observed daily average price | Excluded |
| `units_sold` | Daily units for a store and category | Target |
| `revenue` | Realized sales revenue | Excluded |

Revenue contains information about the sales outcome and would leak information into the model. The observed average price is also excluded because its availability before the selling day has not been established. A verified, planned price could be added in a later exercise.


In [ ]:
DATA_URL = 'https://raw.githubusercontent.com/jrmst102/mancino/main/data/v1_2025-11-01/mancino_daily_sales.csv'
LOCAL_CSV = Path('mancino_daily_sales.csv')

if LOCAL_CSV.exists():
    df = pd.read_csv(LOCAL_CSV)
    print('Loaded local CSV:', LOCAL_CSV)
else:
    try:
        df = pd.read_csv(DATA_URL)
        print('Loaded CSV from GitHub.')
    except Exception as error:
        print('GitHub download failed. Upload mancino_daily_sales.csv when prompted.')
        from google.colab import files
        uploaded = files.upload()
        if 'mancino_daily_sales.csv' not in uploaded:
            raise ValueError('Please upload the named Mancino CSV.') from error
        df = pd.read_csv(io.BytesIO(uploaded['mancino_daily_sales.csv']))

required = {'date', 'store_id', 'category', 'units_sold'}
assert required.issubset(df.columns), 'The CSV is missing required columns.'
df['date'] = pd.to_datetime(df['date'], errors='raise')
assert not df[list(required)].isna().any().any(), 'Check missing required values.'
assert not df.duplicated(['date', 'store_id', 'category']).any(), 'Check duplicate rows.'
assert pd.api.types.is_numeric_dtype(df['units_sold'])
assert df['units_sold'].ge(0).all(), 'Check negative unit sales.'
df = df.sort_values(['date', 'store_id', 'category']).reset_index(drop=True)
print(f'{len(df):,} rows | {df.store_id.nunique()} stores | {df.category.nunique()} categories')
print('Dates:', df.date.min().date(), 'to', df.date.max().date())
display(df.head())
display(df.isna().sum().rename('Missing values').to_frame())


## 3. Define predictors and split by date
Use only information available before the selling day: store, category, weekday, and day of year. Store numbers are identifiers, so they are treated as categories. Day of year supplies a simple numeric time feature; it is not a full seasonality model.

The earliest 60% of dates form the training set, the next 20% form the validation set, and the latest 20% form the test set. Every observation from a given date stays in the same set. This asks whether patterns learned from earlier dates carry forward to later dates for the existing stores and categories.

Train on the training set, choose settings using validation results, and open the test set only after those choices are complete. This notebook does not use lagged sales, promotions, or inventory data.


In [ ]:
df['store_id'] = df['store_id'].astype(str)
df['weekday'] = df['date'].dt.day_name()
df['day_of_year'] = df['date'].dt.dayofyear
CATEGORICAL = ['store_id', 'category', 'weekday']
NUMERIC = ['day_of_year']
FEATURES = CATEGORICAL + NUMERIC
TARGET = 'units_sold'

dates = np.sort(df['date'].unique())
train_end = dates[int(len(dates) * 0.60) - 1]
validation_end = dates[int(len(dates) * 0.80) - 1]
train = df.loc[df.date <= train_end].copy()
validation = df.loc[(df.date > train_end) & (df.date <= validation_end)].copy()
test = df.loc[df.date > validation_end].copy()
assert train.date.max() < validation.date.min() <= validation.date.max() < test.date.min()

X_train, y_train = train[FEATURES], train[TARGET]
X_val, y_val = validation[FEATURES], validation[TARGET]
X_test, y_test = test[FEATURES], test[TARGET]
display(pd.DataFrame([
    {'Set': name, 'Rows': len(part), 'First date': part.date.min().date(),
     'Last date': part.date.max().date()}
    for name, part in [('Training', train), ('Validation', validation), ('Test', test)]
]))


In [ ]:
# Explore training outcomes only; keep later outcomes for evaluation.
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(y_train, bins=30, color='#356DA8', edgecolor='white')
axes[0].set(xlabel='Daily units per store/category', ylabel='Number of observations',
            title='Training sales distribution')
train.groupby('date').units_sold.sum().plot(ax=axes[1], color='#356DA8')
axes[1].set(xlabel='Date', ylabel='Total units across stores/categories',
            title='Training sales over time')
plt.tight_layout()
plt.show()


## 4. Prepare the neural network workflow
One-hot encoding turns categories into indicator columns. Standardizing day of year helps the neural network learn without letting the numeric predictor's larger raw scale dominate training. Indicator columns remain 0 or 1.

The target is also standardized inside `TransformedTargetRegressor`. Its predictions are automatically converted back to units sold. All reported MAE and RMSE values therefore remain in units per store/category/day.

Each network configuration gets a fresh pipeline. The encoder, numeric scaler, and target scaler are fitted within the training period only. Neither the external chronological validation set nor the test set is used to fit these transformations.

| Metric | Interpretation | Preferred direction |
|---|---|---|
| MAE | Average absolute error, in units per store/category/day | Lower |
| RMSE | Error in units, with larger misses receiving more weight | Lower |
| R² | Performance relative to predicting the evaluation set's mean | Higher |

R² can be negative. It is not a percentage of predictions that are correct. Use external validation MAE to select the network architecture. Training loss, early-stopping scores, and selection metrics have different roles.


In [ ]:
def make_pipeline(estimator):
    preparation = ColumnTransformer([
        ('categories', OneHotEncoder(handle_unknown='ignore', sparse_output=False), CATEGORICAL),
        ('numeric', StandardScaler(), NUMERIC)
    ])
    return Pipeline([('prepare', preparation), ('model', estimator)])

def metrics(actual, predicted):
    return {'MAE': mean_absolute_error(actual, predicted),
            'RMSE': np.sqrt(mean_squared_error(actual, predicted)),
            'R2': r2_score(actual, predicted)}

# Rerunning this cell clears previous model runs.
models, results = {}, {}

def fit_and_evaluate(name, estimator):
    pipeline = make_pipeline(estimator)
    started = time.perf_counter()
    pipeline.fit(X_train, y_train)
    seconds = time.perf_counter() - started
    train_scores = metrics(y_train, pipeline.predict(X_train))
    val_scores = metrics(y_val, pipeline.predict(X_val))
    models[name] = pipeline
    results[name] = {'Model': name, 'Train MAE': train_scores['MAE'],
                     'Validation MAE': val_scores['MAE'],
                     'Validation RMSE': val_scores['RMSE'],
                     'Validation R2': val_scores['R2'], 'Fit seconds': seconds}
    display(pd.DataFrame([results[name]]).round(3))
    return pipeline


## 5. Establish a baseline
The baseline always predicts the training set's mean sales. A more complicated model must earn its place by improving on this simple rule. A later extension could use historical averages for each store, category, and weekday as a stronger baseline.


In [ ]:
baseline = fit_and_evaluate('Mean baseline', DummyRegressor(strategy='mean'))


## 6. Neural network regression
This multilayer perceptron combines weighted inputs through hidden layers to predict sales. During training, it adjusts weights and biases to reduce its loss. ReLU activation functions allow the hidden layers to represent nonlinear patterns; the output is a single numeric prediction.

The cell below compares three architectures while keeping the other settings fixed.

| Hidden layers | Structure |
|---|---|
| `(16,)` | One hidden layer with 16 neurons |
| `(32,)` | One hidden layer with 32 neurons |
| `(32, 16)` | Two hidden layers with 32 and 16 neurons |

`max_iter=300` caps training at 300 epochs. An epoch is one pass through the observations used to update weights. Adam is the optimizer, `learning_rate_init=0.001` sets its initial step size, and `alpha=0.01` penalizes large weights. Mini-batches contain 256 observations. A fixed random seed supports reproducibility in the same environment; results can differ with another seed or library version.

### Early stopping and the two validation roles
For each network, early stopping sets aside a random 15% of the training-period rows. This internal holdout monitors R² and can stop training after 20 epochs without sufficient improvement. The fitted network restores the weights with the best internal score. The preprocessing transformations have already been fitted on the training period, including this internal holdout, so this score is a training diagnostic, not an independent performance estimate.

The later, chronological validation set from section 3 is separate. Use its MAE to choose among architectures. The final test period remains untouched. The internal random holdout does not measure future-period performance.

Does a wider or deeper network improve external validation MAE? If a convergence warning appears, record it and consider increasing `MAX_ITER` before opening the test set. Do not suppress the warning.


In [ ]:
HIDDEN_LAYER_OPTIONS = [(16,), (32,), (32, 16)]
MAX_ITER = 300
LEARNING_RATE = 0.001
ALPHA = 0.01
assert HIDDEN_LAYER_OPTIONS and all(
    len(layers) > 0 and all(isinstance(size, int) and size > 0 for size in layers)
    for layers in HIDDEN_LAYER_OPTIONS)
assert MAX_ITER > 0 and LEARNING_RATE > 0 and ALPHA >= 0

# Remove previous network runs when this cell is rerun; retain the baseline.
for name in list(models):
    if name.startswith('Network ('):
        models.pop(name)
        results.pop(name)

for layers in HIDDEN_LAYER_OPTIONS:
    name = f'Network (hidden={layers})'
    estimator = TransformedTargetRegressor(
        regressor=MLPRegressor(
            hidden_layer_sizes=layers, activation='relu', solver='adam',
            alpha=ALPHA, batch_size=256, learning_rate_init=LEARNING_RATE,
            max_iter=MAX_ITER, early_stopping=True, validation_fraction=0.15,
            n_iter_no_change=20, tol=0.0001, random_state=SEED),
        transformer=StandardScaler())
    pipeline = fit_and_evaluate(name, estimator)
    fitted_network = pipeline.named_steps['model'].regressor_
    results[name]['Epochs run'] = fitted_network.n_iter_
    results[name]['Parameters'] = sum(
        array.size for array in fitted_network.coefs_ + fitted_network.intercepts_)
    print('Epochs run:', fitted_network.n_iter_,
          '| Parameters:', results[name]['Parameters'])
    if fitted_network.n_iter_ >= MAX_ITER:
        print('Reached the epoch cap. Inspect the curves before deciding whether to train longer.')


## 7. Compare validation results and select architecture
Compare each network with the baseline. A positive improvement percentage means lower MAE than the baseline; a negative value means worse performance. The code selects the network with the lowest external validation MAE. If there is a tie, the network with fewer learned parameters is preferred.

Compare training error, validation error, fitting time, and parameter count. A larger network can fit more complex patterns but does not guarantee better predictions. A wide training/validation gap can indicate overfitting or a change over time. Training MAE here covers all training-period rows, including the internal early-stopping holdout.

If no network beats the baseline on external validation data, retain the baseline as the preferred approach. Explain which architecture performed best and whether its performance justifies the added complexity. If you change any settings, rerun sections 6–8 before the final test.


In [ ]:
assert 'Mean baseline' in results, 'Run the baseline cell first.'
network_names = [name for name in results if name.startswith('Network (')]
assert network_names, 'Run the neural network exercise first.'

comparison = pd.DataFrame(results.values()).sort_values(
    'Validation MAE', kind='stable').reset_index(drop=True)
baseline_mae = results['Mean baseline']['Validation MAE']
comparison['MAE improvement vs baseline (%)'] = (
    100 * (1 - comparison['Validation MAE'] / baseline_mae)
    if baseline_mae > 0 else np.nan)
display(comparison.round(3).fillna('N/A'))

selected_name = min(network_names, key=lambda name: (
    results[name]['Validation MAE'], results[name]['Parameters']))
selected_layers = models[selected_name].named_steps['model'].regressor_.hidden_layer_sizes
preferred_name = (selected_name if results[selected_name]['Validation MAE'] < baseline_mae
                  else 'Mean baseline')
print('Selected neural network:', selected_name)
print('Preferred approach using external validation MAE:', preferred_name)

comparison.set_index('Model')[['Train MAE', 'Validation MAE']].plot.barh(
    color=['#A7C4DE', '#356DA8'])
plt.xlabel('MAE: units per store/category/day')
plt.title('Training and external validation errors by architecture')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()


## 8. Interpret the training curves
The left panel shows the selected network's training loss by epoch. This is the optimization objective on the standardized target scale, including regularization; it is not MAE in units sold. A falling curve means the optimizer is reducing this objective.

The right panel shows R² on the internal random early-stopping holdout within the training period. Higher is better. The marked epoch has the highest internal score, and its weights are restored when fitting finishes. The last point on the loss curve may therefore describe different weights from those used for predictions.

Do the curves flatten? Does training loss keep decreasing after the internal score stops improving? Explain why low training loss is insufficient evidence of useful predictions on future dates. The chronological validation MAE in section 7, rather than this internal curve, determines which architecture is selected.


In [ ]:
assert 'selected_name' in globals(), 'Run section 7 first.'
selected_network = models[selected_name].named_steps['model'].regressor_
epochs = np.arange(1, len(selected_network.loss_curve_) + 1)
internal_scores = np.asarray(selected_network.validation_scores_)
best_epoch = int(np.argmax(internal_scores)) + 1

print('Selected architecture:', selected_network.hidden_layer_sizes)
print('Epochs run:', selected_network.n_iter_, '| Best internal-score epoch:', best_epoch)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(epochs, selected_network.loss_curve_, color='#356DA8')
axes[0].set(xlabel='Epoch', ylabel='Training loss (standardized target)',
            title='Optimization during training')
axes[1].plot(epochs, internal_scores, color='#356DA8')
axes[1].axvline(best_epoch, color='#B64B44', linestyle='--', label='Best internal R²')
axes[1].set(xlabel='Epoch', ylabel='Internal holdout R²',
            title='Early-stopping diagnostic')
axes[1].legend()
plt.tight_layout()
plt.show()


## 9. Final test: open only after selection
The switch below is off by default. After selecting the architecture, change it to `True`. The code refits the selected pipeline on training plus validation data, with a fresh internal early-stopping holdout drawn from that combined period. It then evaluates the network on the untouched test period. The mean-sales baseline is also refitted on the combined period.

The architecture choice uses external validation results only. Refitting changes the scalers, weights, and possibly the number of training epochs. Section 8 describes the earlier training-period fit, not the final refit.

Do not tune settings after seeing these test results. If you do, the period becomes additional validation data and you need a new test period. Predictions are returned in original sales units and remain unrounded for evaluation. The numeric output is unconstrained and can be negative; the code reports any such estimates, and metrics use the raw predictions.

Complete this interpretation: “With hidden layers ___, the network's test MAE was ___ units per store/category/day, compared with ___ for the baseline. For replenishment planning, this means ___.” These predictions describe synthetic sales patterns; they do not establish causal effects of business decisions.


In [ ]:
RUN_FINAL_TEST = False        # Change to True only after selecting architecture in section 7.

if RUN_FINAL_TEST:
    assert 'selected_name' in globals(), 'Run section 7 to select architecture first.'
    assert selected_name in models, 'Rerun section 7 after changing network settings.'
    X_development = pd.concat([X_train, X_val])
    y_development = pd.concat([y_train, y_val])

    final_network = clone(models[selected_name]).fit(X_development, y_development)
    test_predictions = final_network.predict(X_test)
    final_baseline = DummyRegressor(strategy='mean').fit(X_development, y_development)
    baseline_predictions = final_baseline.predict(X_test)

    print('Network architecture selected using validation MAE:', selected_name)
    print('Approach preferred before opening the test set:', preferred_name)
    test_results = pd.DataFrame([
        {'Model': selected_name, **metrics(y_test, test_predictions)},
        {'Model': 'Refitted mean baseline', **metrics(y_test, baseline_predictions)}
    ])
    display(test_results.round(3))
    print('Negative test predictions:', int(np.sum(test_predictions < 0)))

    prediction_examples = test[['date', 'store_id', 'category', TARGET]].copy()
    prediction_examples['predicted_units'] = test_predictions
    prediction_examples['error_actual_minus_predicted'] = y_test.to_numpy() - test_predictions
    display(prediction_examples.head(10).round(2))

    plt.scatter(y_test, test_predictions, alpha=0.15, s=12, color='#356DA8')
    low = min(y_test.min(), test_predictions.min())
    high = max(y_test.max(), test_predictions.max())
    plt.plot([low, high], [low, high], color='#B64B44', linestyle='--')
    plt.xlabel('Actual units')
    plt.ylabel('Predicted units')
    plt.title(f'Final test: {selected_name}')
    plt.tight_layout()
    plt.show()
else:
    print('Test results remain unopened. Compare validation results and select architecture first.')
